In [ ]:
import datetime
import pandas as pd

In [ ]:
data = pd.read_excel('data/Essentials 4x Logbook.xlsx', sheet_name='4x Program')
data.head().T

In [ ]:
def col_renamer(col):
    col_renamed = (col
        .lower()
        .replace(' ', '_')
        .replace('\n', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('-', '')
    )
    return col_renamed

dropped_cols = ['prior_load',
                'prior_reps',
                'prior_load.1',
                'prior_row',
                '2xprior_load',
                '2xprior_reps',
                '2xprior_load.1',
                'prior_row.1',
                "jeff's_notes"]

dtype_changes = dict(
    # working_sets=int,
)

df = (data.rename(columns=col_renamer)
      .drop(dropped_cols, axis=1)
      .astype(dtype_changes)
     )

# Example column name: "Day"
# Each contiguous block of the same Day value is one workout session
session_number = df["day"].ne(df["day"].shift()).cumsum() - 1

workout_frequency = pd.offsets.CustomBusinessDay(
    weekmask="Mon Tue Thu Fri"
)

# Generate one date per distinct workout session
session_dates = pd.date_range(
    start="2025-03-17",
    periods=session_number.max() + 1,
    freq=workout_frequency
)

# Assign the same date to every exercise in that session
df.insert(
    0,
    "date_day",
    session_number.map(dict(enumerate(session_dates)))
)

df.head().T

In [ ]:
assert df[['day', 'date_day']].value_counts().shape[0] == 4 * 12 # 12 weeks, 4 workouts per day
df[['day', 'date_day']].drop_duplicates().head(6).T

## Parsing the `load` column

Grammar observed in the raw cells:

| Piece | Example | Meaning |
|---|---|---|
| `{weight}R{reps}` | `55R5` | one working set |
| `,` or `\n` between entries | `95R10,\n105R9` | separate working sets (weight and/or reps changed) |
| `▼` or `DS` before an entry | `80R10\n▼40R12`, `45R12DS25R12` | drop set hung off the previous working set |
| `xN` suffix | `160R12x2` | that same set was repeated N times |
| single entry, `working_sets > 1` | `100R10` with `working_sets=2` | shorthand: every working set was at that load |

Reps are matched as **at most 2 digits**, which makes a missing separator
(`115R11105R11`) tokenise correctly as `115R11` + `105R11` instead of `reps=11105`.


In [ ]:
import re

# Optional drop marker (▼ or DS), weight, "R", reps, optional "xN" repeat count.
SET_RE = re.compile(
    r'(?P<drop>▼|DS)?\s*'
    r'(?P<weight>\d+(?:\.\d+)?)\s*R\s*(?P<reps>\d{1,2})'
    r'(?:\s*[xX]\s*(?P<repeat>\d+))?',
    re.IGNORECASE,
)
SEPARATORS = set(',\n\r\t ▼')


def parse_load(raw, declared_sets=None):
    """Parse one `load` cell into a list of set dicts plus a parse status.

    Each set is {set_num, kind, drop_num, weight, reps, inferred}. On a drop set,
    `set_num` points at the working set it hung off of. `inferred` marks sets that
    were filled in from `declared_sets` rather than written down.

    Status is one of: ok / partial (leftover text) / weight_only (bare number,
    no reps) / unparsed / empty.
    """
    # `load` is a str-dtype column, so a blank cell arrives as pd.NA, not float nan.
    if raw is None or pd.isna(raw):
        return [], 'empty'

    text = str(raw).strip()
    if not text:
        return [], 'empty'

    sets, spans = [], []
    set_num = drop_num = 0

    for m in SET_RE.finditer(text):
        spans.append(m.span())
        weight, reps = float(m.group('weight')), int(m.group('reps'))
        for _ in range(int(m.group('repeat') or 1)):
            if m.group('drop'):
                drop_num += 1
                kind, num = 'drop', set_num
            else:
                set_num, drop_num = set_num + 1, 0
                kind, num = 'working', set_num
            sets.append(dict(set_num=num, kind=kind, drop_num=drop_num,
                             weight=weight, reps=reps, inferred=False))

    # Anything left over that isn't a separator means we didn't understand the cell.
    covered = set(i for start, end in spans for i in range(start, end))
    leftover = ''.join(c for i, c in enumerate(text)
                       if i not in covered and c not in SEPARATORS)

    if not sets:
        if re.fullmatch(r'\d+(?:\.\d+)?', text):  # bare number: weight logged, reps never were
            return [dict(set_num=1, kind='working', drop_num=0,
                         weight=float(text), reps=pd.NA, inferred=False)], 'weight_only'
        return [], 'unparsed'

    status = 'ok' if not leftover else 'partial'

    # Shorthand: one entry logged but the program called for more sets -> repeat it.
    # Any drop set stays attached to the final working set only.
    if declared_sets is not None and not pd.isna(declared_sets):
        working = [s for s in sets if s['kind'] == 'working']
        missing = int(declared_sets) - len(working)
        if len(working) == 1 and missing > 0:
            template = working[0]
            extra = [dict(template, set_num=template['set_num'] + i, inferred=True)
                     for i in range(1, missing + 1)]
            drops = [dict(s, set_num=template['set_num'] + missing)
                     for s in sets if s['kind'] == 'drop']
            sets = working + extra + drops

    return sets, status

In [ ]:
ID_COLS = ['date_day', 'day', 'exercise']


def build_sets(df, load_col='load', sets_col='working_sets', id_cols=ID_COLS):
    """Tidy frame: one row per logged set."""
    rows = []
    for idx, raw in df[load_col].items():
        parsed, status = parse_load(raw, df.at[idx, sets_col] if sets_col in df else None)
        base = {'row_id': idx, **{c: df.at[idx, c] for c in id_cols if c in df}}
        if not parsed:
            parsed = [dict(set_num=pd.NA, kind=pd.NA, drop_num=pd.NA,
                           weight=pd.NA, reps=pd.NA, inferred=False)]
        rows += [{**base, **s, 'parse_status': status, 'load_raw': raw} for s in parsed]

    return (pd.DataFrame(rows)
            .sort_values(['row_id', 'set_num', 'drop_num'])
            .reset_index(drop=True)
            .assign(volume_load=lambda d: d.weight * d.reps))


sets = build_sets(df)
sets.head(12)

In [ ]:
def summarize_sets(sets):
    """Collapse the tidy set frame back to one row per exercise-row."""
    work = sets[sets.kind == 'working']
    drop = sets[sets.kind == 'drop']

    summary = pd.DataFrame({
        'parse_status': sets.groupby('row_id').parse_status.first(),
        'n_working_sets': work.groupby('row_id').size(),
        'n_drop_sets': drop.groupby('row_id').size(),
        'weights': work.groupby('row_id').weight.apply(tuple),
        'weight_changed': work.groupby('row_id').weight.nunique() > 1,
        'weight_first': work.groupby('row_id').weight.first(),
        'weight_top': work.groupby('row_id').weight.max(),
        'reps_total': work.groupby('row_id').reps.sum(min_count=1),
        'volume_load': work.groupby('row_id').volume_load.sum(min_count=1),
        'volume_load_incl_drops': sets.groupby('row_id').volume_load.sum(min_count=1),
        'drop_weight': drop.groupby('row_id').weight.first(),
        'drop_reps': drop.groupby('row_id').reps.first(),
        'sets_inferred': work.groupby('row_id').inferred.any(),
    })
    # reps recorded at the heaviest working set
    top = (work.sort_values(['weight', 'set_num'])
               .groupby('row_id').last()[['reps']].rename(columns={'reps': 'reps_at_top'}))

    return (summary.join(top)
                   .assign(n_drop_sets=lambda d: d.n_drop_sets.fillna(0).astype(int),
                           n_working_sets=lambda d: d.n_working_sets.fillna(0).astype(int),
                           has_dropset=lambda d: d.n_drop_sets > 0)
                   .rename_axis(None))


df_parsed = df.join(summarize_sets(sets))
df_parsed[['exercise', 'load', 'working_sets', 'n_working_sets', 'weights',
           'weight_changed', 'weight_top', 'reps_at_top', 'volume_load',
           'has_dropset', 'drop_weight', 'drop_reps', 'parse_status']].head(25)

In [ ]:
# QA: everything that isn't a clean parse, plus any set-count that still disagrees
# with the program's `working_sets`. Fix these in the spreadsheet, not in code.
print(df_parsed.parse_status.value_counts().to_string(), '\n')

flagged = df_parsed[(df_parsed.parse_status != 'ok')
                    | (df_parsed.n_working_sets != df_parsed.working_sets)]
flagged[['exercise', 'load', 'working_sets', 'n_working_sets', 'parse_status']]

## Data dictionary — `df_parsed`

288 rows, one per programmed exercise-row. Index is the original spreadsheet row order.

### Prescribed / logged (from the spreadsheet)

| Column | dtype | Description |
|---|---|---|
| `date_day` | `datetime64` | **Synthetic** workout date — a Mon/Tue/Thu/Fri sequence from 2025-03-17, one row per day. Not the real session date, and it spreads a single session's exercises across separate days. |
| `day` | `str` | Program slot, `W{week}U\|L{session}` — e.g. `W1U1` = week 1, upper 1. 48 slots. |
| `exercise` | `str` | Movement name. Contains embedded `\n` (`'Flat DB Press\n(Heavy)'`); an `A1:`/`A2:` prefix marks a superset pair. |
| `warmup_sets` | `object` | Prescribed warm-up sets. (e.g. 1-2)
| `working_sets` | `int64` | Prescribed working sets (1–3). Ground truth for the shorthand expansion. |
| `reps` | `str` | Prescribed **rep range**, e.g. `'10-12'`. May carry `'(drop set)'` or `'per leg'`. Not what you actually did — that's in `load`. |
| `load` | `str` | Raw logged load string. Source for every derived column below. |
| `rpe` | `str` | Prescribed RPE range: `'8-9'`, `'9-10'`, `'10'`. |
| `rest_min` | `object` | Prescribed rest, minutes. Mostly numeric, but `'1.5 loop'` appears. |
| `notes` | `str` | Free text, 27 non-null. |
| `sub_1`, `sub_2` | `str` | Prescribed substitute exercises. |

### Derived from `load`

| Column | dtype | Description |
|---|---|---|
| `parse_status` | `str` | `ok` / `partial` (leftover text) / `weight_only` (bare number, no reps) / `unparsed` / `empty`. Currently `ok` for all 288 rows. |
| `n_working_sets` | `int64` | Working sets recovered, after shorthand expansion. Currently equals `working_sets` on every row. |
| `n_drop_sets` | `int64` | Drop sets recovered (0 or 1). |
| `weights` | `object` | Tuple of each working set's weight, in order — the lossless view. |
| `weight_changed` | `bool` | More than one distinct weight across working sets (14 rows). |
| `weight_first` | `float64` | Weight of set 1. |
| `weight_top` | `float64` | Heaviest working set. **Use for top-end strength.** |
| `reps_at_top` | `int64` | Reps at `weight_top` (last set at that weight if tied). |
| `reps_total` | `int64` | Sum of working-set reps. |
| `volume_load` | `float64` | Σ weight×reps over working sets. **Main progression metric** — the only summary that stays honest when weight changes mid-exercise. |
| `volume_load_incl_drops` | `float64` | Same, including drop sets. Kept separate since drop volume isn't comparable to working volume. |
| `has_dropset` | `bool` | Any drop set logged (45 rows). |
| `drop_weight` | `float64` | Weight of the **first** drop set; `NaN` if none. No row logs more than one drop, so nothing is lost today — if that changes, use the `sets` frame. |
| `drop_reps` | `float64` | Reps of the first drop set; `NaN` if none. |
| `sets_inferred` | `bool` | `True` where sets were filled in from the `working_sets` shorthand rather than written out individually (194 rows). Exclude these if you need only hand-recorded sets. |

Units are whatever the machine/dumbbell was marked in — **not consistent across exercises**, so only compare a given `exercise` to itself over time. `weight = 0` means bodyweight.

The per-set frame `sets` (557 rows) carries the same derived fields at one-row-per-set granularity, keyed by `row_id` → `df_parsed` index.


In [ ]:
df.columns